# Silver — ecommerce_rastreamento_entregas

Este notebook lê a camada Bronze de rastreamento, aplica as 10 regras de qualidade/negócio, salva a tabela Silver em Delta e registra o resumo das falhas em `squad1.dq_monitoring_logs`.

In [0]:



from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from functools import reduce
import uuid
import pyarrow as pa
from deltalake import DeltaTable
from deltalake.writer import write_deltalake
from azure.core.exceptions import ResourceNotFoundError

RUN_ID = str(uuid.uuid4())

# Flags de execução
FORCAR_REPROCESSAMENTO = True
SOBRESCREVER_SILVER = True

CONTAINER_DESTINO = "squad1"
TABELA_DQ_LOGS = "squad1.dq_monitoring_logs"

container_squad1 = service_client.get_file_system_client("squad1")

print("RUN_ID:", RUN_ID)

ENTIDADE = "ecommerce_rastreamento"
CAMINHO_BRONZE = "az://squad1/bronze/ecommerce_rastreamento"
CAMINHO_BRONZE_PEDIDOS = "az://squad1/bronze/ecommerce_pedidos"
CAMINHO_SILVER = "az://squad1/silver/ecommerce_rastreamento"
PASTA_SILVER = "silver/ecommerce_rastreamento"
NOME_TABELA_DQ = "ecommerce_rastreamento"
print("Bronze:", CAMINHO_BRONZE)
print("Silver:", CAMINHO_SILVER)


##  FUNÇÕES AUXILIARES


In [0]:

def delta_existe(caminho_delta):
    try:
        DeltaTable(caminho_delta, storage_options=storage_options)
        return True
    except Exception:
        return False


def ler_delta_spark(caminho_delta):
    dt = DeltaTable(caminho_delta, storage_options=storage_options)
    pdf = dt.to_pyarrow_table().to_pandas()
    if len(pdf) == 0:
        raise Exception(f"Delta existe, mas está vazio: {caminho_delta}")
    return spark.createDataFrame(pdf)


def salvar_delta_spark(df, caminho_delta, mode="append", partition_by=None, pasta_relativa=None):
    qtd = df.count()
    if qtd == 0:
        print(f"Gravação ignorada em {caminho_delta}: DataFrame vazio.")
        return

    if mode == "overwrite" and pasta_relativa:
        try:
            container_squad1.delete_directory(pasta_relativa)
            print(f"Delta antigo removido: {pasta_relativa}")
        except ResourceNotFoundError:
            print(f"Delta antigo não existia: {pasta_relativa}")
        except Exception as e:
            print(f"Aviso ao remover {pasta_relativa}: {e}")

    table = pa.Table.from_pandas(df.toPandas(), preserve_index=False)

    kwargs = {
        "table_or_uri": caminho_delta,
        "data": table,
        "mode": mode,
        "storage_options": storage_options
    }
    if partition_by:
        kwargs["partition_by"] = partition_by

    write_deltalake(**kwargs)
    print(f"Delta gravado em {caminho_delta} | modo={mode} | registros={qtd}")


def anti_duplicidade_por_arquivo(df_novo, caminho_delta_destino, coluna_arquivo="bronze_source_file"):
    if FORCAR_REPROCESSAMENTO:
        print("FORCAR_REPROCESSAMENTO=True: todos os registros da Bronze serão avaliados novamente.")
        return df_novo

    if delta_existe(caminho_delta_destino):
        df_destino = ler_delta_spark(caminho_delta_destino)
        if coluna_arquivo in df_destino.columns:
            arquivos_processados = df_destino.select(coluna_arquivo).dropDuplicates()
            return df_novo.join(arquivos_processados, on=coluna_arquivo, how="left_anti")
    return df_novo


schema_dq_logs = StructType([
    StructField("run_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("regra", StringType(), False),
    StructField("status", StringType(), False),
    StructField("severidade", StringType(), False),
    StructField("qtd_registros_falhos", IntegerType(), True),
    StructField("qtd_registros_total", IntegerType(), False),
    StructField("timestamp_execucao", TimestampType(), False),
    StructField("arquivo_origem", StringType(), True),
])

COLUNAS_DQ_LOGS = [f.name for f in schema_dq_logs.fields]


def padronizar_schema_dq_logs(df):
    return (
        df.select(*COLUNAS_DQ_LOGS)
        .withColumn("run_id", F.col("run_id").cast("string"))
        .withColumn("tabela", F.col("tabela").cast("string"))
        .withColumn("regra", F.col("regra").cast("string"))
        .withColumn("status", F.col("status").cast("string"))
        .withColumn("severidade", F.col("severidade").cast("string"))
        .withColumn("qtd_registros_falhos", F.col("qtd_registros_falhos").cast("int"))
        .withColumn("qtd_registros_total", F.col("qtd_registros_total").cast("int"))
        .withColumn("timestamp_execucao", F.col("timestamp_execucao").cast("timestamp"))
        .withColumn("arquivo_origem", F.col("arquivo_origem").cast("string"))
    )


def gerar_logs_por_regras(df_validado, regras, tabela_nome, qtd_total):
    logs = []
    for regra in regras:
        coluna = regra["coluna"]
        nome = regra["regra"]
        severidade = regra.get("severidade", "Critica")

        if coluna not in df_validado.columns:
            print(f"Aviso: coluna de regra não encontrada: {coluna}")
            continue

        df_log = (
            df_validado
            .filter(F.col(coluna) == True)
            .groupBy("bronze_source_file")
            .agg(F.count(F.lit(1)).cast("int").alias("qtd_registros_falhos"))
            .withColumn("run_id", F.lit(RUN_ID))
            .withColumn("tabela", F.lit(tabela_nome))
            .withColumn("regra", F.lit(nome))
            .withColumn("status", F.lit("FAIL"))
            .withColumn("severidade", F.lit(severidade))
            .withColumn("qtd_registros_total", F.lit(int(qtd_total)).cast("int"))
            .withColumn("timestamp_execucao", F.current_timestamp())
            .withColumnRenamed("bronze_source_file", "arquivo_origem")
            .select(*COLUNAS_DQ_LOGS)
        )
        logs.append(df_log)

    if not logs:
        return spark.createDataFrame([], schema_dq_logs)

    return reduce(lambda a, b: a.unionByName(b), logs)


def gravar_dq_logs(df_logs):
    df_logs = padronizar_schema_dq_logs(df_logs)
    qtd = df_logs.count()
    print("Logs novos para dq_monitoring_logs:", qtd)
    if qtd == 0:
        print("Nenhum log novo para gravar.")
        return

    (
        df_logs
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TABELA_DQ_LOGS)
    )
    print(f"Logs gravados na tabela {TABELA_DQ_LOGS}.")


def safe_read_delta(caminho_delta, nome):
    try:
        df = ler_delta_spark(caminho_delta)
        print(f"{nome}: {df.count()} registros")
        return df
    except Exception as e:
        print(f"Aviso: não foi possível ler {nome} em {caminho_delta}: {e}")
        return None


## Ler Bronze e referências

In [0]:
df_bronze_pedidos = ler_delta_spark(CAMINHO_BRONZE_PEDIDOS)

if "bronze_source_file" not in df_bronze_pedidos.columns:
    raise Exception("A Bronze precisa conter a coluna bronze_source_file.")

if "bronze_ingested_at" not in df_bronze_pedidos.columns:
    raise Exception("A Bronze precisa conter a coluna bronze_ingested_at.")

if FORCAR_REPROCESSAMENTO:
    print("Reprocessamento forçado ativado: toda a Bronze será reavaliada.")
    df_micro_lote = df_bronze_pedidos
else:
    df_micro_lote = anti_duplicidade_por_arquivo(
        df_novo=df_bronze_pedidos,
        caminho_delta_destino=CAMINHO_SILVER_PEDIDOS,
        coluna_arquivo="bronze_source_file"
    )

qtd_micro_lote = df_micro_lote.count()
TEM_MICRO_LOTE_NOVO = qtd_micro_lote > 0

print("Registros para processar:", qtd_micro_lote)

if TEM_MICRO_LOTE_NOVO:
    display(
        df_micro_lote
        .select("bronze_source_file")
        .dropDuplicates()
        .orderBy("bronze_source_file")
    )
else:
    print("Nenhum registro para processar na Silver de pedidos.")

## Aplicar 10 regras

In [0]:

if TEM_MICRO_LOTE_NOVO:
    status_validos = ["em separacao", "coletado", "em transito", "saiu para entrega", "entregue"]
    status_rank_expr = (
        F.when(F.col("status_entrega_norm") == "em separacao", 1)
        .when(F.col("status_entrega_norm") == "coletado", 2)
        .when(F.col("status_entrega_norm") == "em transito", 3)
        .when(F.col("status_entrega_norm") == "saiu para entrega", 4)
        .when(F.col("status_entrega_norm") == "entregue", 5)
    )

    w_rast = Window.partitionBy("id_rastreamento")
    w_pedido_status = Window.partitionBy("id_pedido_ecommerce").orderBy("status_rank", "dt_evento_ts")

    df_base = (
        df_micro_lote
        .withColumn("status_entrega_norm", F.lower(F.trim(F.col("status_entrega").cast("string"))))
        .withColumn("dt_evento_ts", F.col("dt_evento").cast("timestamp"))
        .withColumn("status_rank", status_rank_expr)
        .withColumn("qtd_id_rastreamento", F.count("*").over(w_rast))
        .join(df_pedidos_ref, on="id_pedido_ecommerce", how="left")
        .withColumn("pedido_existe", F.col("status_pedido").isNotNull())
    )

    df_por_pedido = (
        df_base
        .groupBy("id_pedido_ecommerce")
        .agg(
            F.max(F.when(F.col("status_entrega_norm") == "entregue", 1).otherwise(0)).alias("tem_evento_entregue"),
            F.min(F.when(F.col("status_entrega_norm") == "coletado", F.col("dt_evento_ts"))).alias("dt_coletado"),
            F.max(F.when(F.col("status_entrega_norm") == "entregue", F.col("dt_evento_ts"))).alias("dt_entregue"),
            F.countDistinct("id_transportadora").alias("qtd_transportadoras")
        )
    )

    df_regras = (
        df_base
        .withColumn("dt_evento_anterior", F.lag("dt_evento_ts").over(w_pedido_status))
        .join(df_por_pedido, on="id_pedido_ecommerce", how="left")
    )

    df_silver = (
        df_regras
        .withColumn("r1_id_rastreamento_falhou", F.col("id_rastreamento").isNull() | (F.col("qtd_id_rastreamento") > 1))
        .withColumn("r2_status_entrega_falhou", F.col("status_entrega_norm").isNull() | (~F.col("status_entrega_norm").isin(status_validos)))
        .withColumn("r3_id_pedido_inexistente_falhou", F.col("id_pedido_ecommerce").isNull() | (F.col("pedido_existe") == False))
        .withColumn("r4_dt_evento_falhou", F.col("dt_evento_ts").isNull() | (F.col("dt_evento_ts") > F.current_timestamp()))
        .withColumn("r5_codigo_rastreio_falhou", F.col("codigo_rastreio").isNull() | (~F.col("codigo_rastreio").cast("string").rlike(r"^[A-Z]{2}[0-9]{9}$")))
        .withColumn("r6_ordem_cronologica_falhou", F.col("dt_evento_anterior").isNotNull() & (F.col("dt_evento_ts") < F.col("dt_evento_anterior")))
        .withColumn("r7_entregue_sem_evento_falhou", (F.col("status_pedido") == "Entregue") & (F.coalesce(F.col("tem_evento_entregue"), F.lit(0)) == 0))
        .withColumn("r8_tempo_entrega_maior_30_falhou", F.col("dt_coletado").isNotNull() & F.col("dt_entregue").isNotNull() & (F.datediff(F.col("dt_entregue"), F.col("dt_coletado")) > 30))
        .withColumn("r9_cancelado_com_evento_falhou", F.col("status_pedido") == "Cancelado")
        .withColumn("r10_transportadora_inconsistente_falhou", (F.col("status_pedido") != "Cancelado") & (F.coalesce(F.col("qtd_transportadoras"), F.lit(0)) != 1))
    )

    regras_dq = [
        {"coluna":"r1_id_rastreamento_falhou", "regra":"R1_ID_RASTREAMENTO_NULO_OU_DUPLICADO", "severidade":"Critica"},
        {"coluna":"r2_status_entrega_falhou", "regra":"R2_STATUS_ENTREGA_INVALIDO", "severidade":"Critica"},
        {"coluna":"r3_id_pedido_inexistente_falhou", "regra":"R3_ID_PEDIDO_INEXISTENTE", "severidade":"Critica"},
        {"coluna":"r4_dt_evento_falhou", "regra":"R4_DT_EVENTO_INVALIDA", "severidade":"Critica"},
        {"coluna":"r5_codigo_rastreio_falhou", "regra":"R5_CODIGO_RASTREIO_INVALIDO", "severidade":"Critica"},
        {"coluna":"r6_ordem_cronologica_falhou", "regra":"R6_ORDEM_CRONOLOGICA_INVALIDA", "severidade":"Critica"},
        {"coluna":"r7_entregue_sem_evento_falhou", "regra":"R7_PEDIDO_ENTREGUE_SEM_EVENTO_ENTREGUE", "severidade":"Critica"},
        {"coluna":"r8_tempo_entrega_maior_30_falhou", "regra":"R8_TEMPO_COLETADO_ENTREGUE_MAIOR_30_DIAS", "severidade":"Critica"},
        {"coluna":"r9_cancelado_com_evento_falhou", "regra":"R9_PEDIDO_CANCELADO_COM_EVENTO", "severidade":"Critica"},
        {"coluna":"r10_transportadora_inconsistente_falhou", "regra":"R10_TRANSPORTADORA_INCONSISTENTE", "severidade":"Critica"},
    ]
    falha_critica = reduce(lambda a,b: a | b, [F.col(r["coluna"]) for r in regras_dq])
    df_silver = df_silver.withColumn("silver_linha_valida", ~falha_critica).withColumn("silver_processed_at", F.current_timestamp()).drop("qtd_id_rastreamento", "pedido_existe")
    display(df_silver.limit(10))
else:
    print("Sem micro-lote novo.")


## Gravar Silver válida e logs

In [0]:

if TEM_MICRO_LOTE_NOVO:
    qtd_total = df_silver.count()
    df_silver_validos = df_silver.filter(F.col("silver_linha_valida") == True)
    df_logs = gerar_logs_por_regras(df_silver, regras_dq, NOME_TABELA_DQ, qtd_total)
    print("Registros válidos para Silver:", df_silver_validos.count())
    print("Logs gerados:", df_logs.count())
    modo_silver = "overwrite" if (FORCAR_REPROCESSAMENTO and SOBRESCREVER_SILVER) else "append"
    salvar_delta_spark(df_silver_validos, CAMINHO_SILVER, mode=modo_silver, pasta_relativa=PASTA_SILVER if modo_silver == "overwrite" else None)
    gravar_dq_logs(df_logs)
else:
    print("Nada para salvar: não há micro-lote novo.")


##  VALIDACAO


In [0]:

print("\n===== Validação final =====")
try:
    print("Silver:", CAMINHO_SILVER)
    df_val_silver = ler_delta_spark(CAMINHO_SILVER)
    print("Registros na Silver:", df_val_silver.count())
    display(df_val_silver.limit(20))
except Exception as e:
    print("Silver ainda não disponível ou vazia:", e)

try:
    df_logs_val = spark.table(TABELA_DQ_LOGS)
    print("Registros totais em dq_monitoring_logs:", df_logs_val.count())
    display(df_logs_val.orderBy(F.col("timestamp_execucao").desc()).limit(30))
except Exception as e:
    print("Não foi possível consultar dq_monitoring_logs:", e)
